# 15. Monte Carlo re-test: per-lesion sources, no halo, 10 compartments, voxel target

Changes to the pilot of notebook 14, on the same 3 phantoms, before the full run:

1. **Lesions.** The cohort draws the release rate per lesion, and a grouped $D_c$ cannot take a per-lesion TIA. The grouped Monte Carlo stays for long range and cross-dose; each lesion's self-dose comes from a VSV on the native PET grid. New GATE kernels for 2.73×2.73×3.27 and 4.07×4.07×5.0 mm, extended to ±200 mm.
2. **Rest-of-body halo.** The PBPK kidney TIA comes from the 10 mm grown mask (notebook 11), so kidney spill-out in the PET-weighted rest of body would count twice. In that weight, the 10 mm shells around kidneys, glands and bladder take the median rest-of-body concentration.
3. **Compartments of the PBPK.** Blood pool and spine become sources: 10 compartments.

Then the **voxel target**: VSV_native/ρ per source and lesion, plus a smooth Monte Carlo long-range term (Monte Carlo minus native VSV, low-pass filtered), compared with a direct 2×10⁶ run.

GATE runs in `scripts/monte_carlo/run_mc_retest.py` (resumable, 4 threads), kernels in `scripts/gate/lu177_native_kernel.py`, code in `src/psma_mc.py`. Outputs in `data/processed/psma/mc_pilot/v2/`, kernels in `output/vsv_native/`. The 7 compartments unchanged since the pilot reuse its $D_c$ on the 3 phantoms. A fourth phantom, on the third grid (`psma_de1f4300eda3bef3_2017-04-23`, 11 lesions, 2 mm slices), has all 10 sources at 6×10⁵ decays, its direct run and 3 single lesions. **The full run is not launched here.**

**About this notebook.** Re-test before the full run: native kernels, per-lesion sources, 10 compartments, voxel target.
- **Inputs:** phantoms, pilot runs; `scripts/monte_carlo/run_mc_retest.py`, `scripts/gate/lu177_native_kernel.py`.
- **Outputs:** `data/processed/psma/mc_pilot/v2/`, `output/vsv_native/`.
- **Run time:** about 65 min with the runs done.
- **Environment:** .venv (opengate).

In [ ]:
import json, sys, time
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

WORKSPACE = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WORKSPACE / "src"))
sys.path.insert(0, str(WORKSPACE / "scripts" / "monte_carlo"))
import psma_mc as M
import run_mc_retest as R

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200, "display.float_format", lambda v: f"{v:.3g}")
plt.rcParams.update({"figure.dpi": 90, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
INK, GRID, BLUE, ORANGE, GREEN = "#52514e", "#e5e4e0", "#2a78d6", "#eb6834", "#1baf7a"
STUDIES = list(R.STUDIES)
THIRD = R.THIRD_GRID
ALL = STUDIES + [THIRD]
NAME = dict(zip(ALL, ["typical", "high TLV", "single kidney", "third grid"]))
COMPARTMENTS = list(M.COMPARTMENTS_V2)
TARGETS = list(M.TARGETS)
NEW = R.NEW
DATA, OLD = M.V2, M.PILOT
QC = DATA / "figures"
QC.mkdir(exist_ok=True)
raw = pd.read_csv(DATA / "targets.csv")
print(raw.drop_duplicates("run_id").groupby(["study_id", "kind"]).agg(runs=("run_id", "size"), seconds=("seconds", "sum")).round(0).to_string())

def read(path):
    return np.asanyarray(nib.load(path).dataobj).astype(np.float64)

def d_c(study, c, variant="pet"):
    folder = DATA if c in NEW or study == THIRD else OLD
    return read(folder / study / f"D_{c}_{variant}.nii.gz")

organs_of = {s: M.load(s, "organs") for s in ALL}
density_of = {s: M.density_map(M.load(s, "ct"), organs_of[s]) for s in ALL}
masks_of = {s: M.target_masks(s) for s in ALL}
spacing_of = {s: M.spacing_of(s) for s in ALL}
body_of = {s: organs_of[s] > 0 for s in ALL}
weighted = lambda s, dose: M.mass_weighted_on_targets(dose, density_of[s], masks_of[s])

@lru_cache(maxsize=1)
def engine(s):
    return M.NativeVsv(organs_of[s].shape, spacing_of[s])

@lru_cache(maxsize=1)
def split_of(s):
    return M.SplitVsv(organs_of[s].shape, spacing_of[s])

water = lambda s, c: engine(s)(M.source_weights_v2(s, c, "pet"))

## 1. Native kernels

Three grids, since the split also holds 4.07×4.07×2.0 mm (Biograph20_mCT, 26 training and 6 test studies). Water cube of 80 cm, Lu-177 uniform in the central voxel, dose scored on the same anisotropic grid over ±200 mm, 2×10⁷ decays, cut 0.1 mm, physics as `lu177_water_validation.py`. Post-processing in `native_kernel`: symmetrised over the voxel symmetries; beyond 10 mm, where electrons are gone and voxels hold noise, the radial profile of the photon tail in 4 mm shells replaces the voxels.

Checks: energy balance against the emitted electrons and photons, and the old 4 mm kernel on the common range.

In [ ]:
rows, profiles = [], {}
old = np.load(M.KERNEL)
EMITTED_E, EMITTED_PH = 0.1479, 0.0351
stem = lambda spacing: "kernel_{:.2f}x{:.2f}x{:.2f}".format(*spacing).replace(".", "p")
MEV_TO_MGY_G_PER_MBQ_S = 1.602176634e-13 * 1e6 * 1e3 * 1000
for spacing in sorted(R.KERNELS):
    k, r = M.native_kernel(spacing)
    raw_k = np.load(M.NATIVE_KERNELS / (stem(spacing) + "_raw.npy"))
    voxel_g = np.prod(spacing) / 1000
    energy = lambda a: float(a.sum() * voxel_g / MEV_TO_MGY_G_PER_MBQ_S)
    info = json.loads((M.NATIVE_KERNELS / (stem(spacing) + ".json")).read_text())
    rows.append(dict(voxel=str(tuple(round(s, 2) for s in spacing)), shape=str(k.shape), decays=info["n"], seconds=round(info["seconds"]), centre_mgy_per_mbq_s=k[tuple(n // 2 for n in k.shape)],
                     energy_total_mev=energy(k), energy_raw_mev=energy(raw_k), energy_within_40mm=energy(k[r <= 40]), energy_beyond_40mm=energy(k[r > 40]), share_of_photon_energy_beyond_40mm=energy(k[r > 40]) / EMITTED_PH,
                     tail_step_at_10mm=k[(r > 10) & (r < 11)].mean() / np.mean(np.load(M.NATIVE_KERNELS / (stem(spacing) + "_raw.npy"))[(r > 9) & (r <= 10)])))
    profiles[spacing] = (r.ravel(), k.ravel())
kernels = pd.DataFrame(rows)
print(kernels.T.to_string())
c = np.array(old.shape) // 2
idx = np.indices(old.shape) - c[:, None, None, None]
r_old = np.sqrt((idx ** 2).sum(0)) * M.KERNEL_MM
print("Old 4 mm kernel: emitted electrons", EMITTED_E, "MeV/decay; energy within its ±40 mm box:", round(float(old.sum() * 0.064 / MEV_TO_MGY_G_PER_MBQ_S), 4), "MeV/decay")

fig, ax = plt.subplots(figsize=(7, 3.4))
for (spacing, (r, k)), color in zip(profiles.items(), [BLUE, ORANGE, GREEN]):
    bins = np.arange(0, 205, 3.0)
    which = np.digitize(r, bins)
    mean = np.bincount(which, weights=k) / np.maximum(np.bincount(which), 1)
    ok = (mean > 0)
    ax.semilogy((bins[np.minimum(np.flatnonzero(ok), len(bins) - 1)]), mean[ok], lw=2, color=color, label=f"{spacing[0]:.2f}x{spacing[1]:.2f}x{spacing[2]:.2f} mm")
ax.semilogy(r_old[r_old > 0].ravel()[::7], old[r_old > 0].ravel()[::7], ".", ms=3, color=INK, label="old 4 mm kernel, ±40 mm")
ax.set(xlabel="distance from the source voxel (mm)", ylabel="mGy/(MBq s)", title="Native kernels: the photon tail continues far beyond the old edge")
ax.grid(color=GRID, lw=.6); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(QC / "kernels.png", dpi=150); plt.show()

**Reading.** All three kernels carry 0.176 MeV/decay and 0.156 within 40 mm, whatever the voxel; emitted are 0.148 in electrons and 0.035 in photons, so the rest leaves the ±200 mm box. **Beyond 40 mm lies 0.020 MeV/decay, 57% of the photon energy**: this is what the old kernel lacked. It agrees with the new ones on the common range (0.1575 within its ±40 mm box). Centre 0.863, 0.640, 0.266 mGy/(MBq s), by voxel volume. Cost 993, 892 and 424 s with another session on the machine. The join at 10 mm drops by 0.68-0.75, against 0.82 from $1/r^2$ alone: no visible step. The fall at 204 mm in the figure is the last, partly filled distance bin.

## 2. Native VSV against Monte Carlo, per compartment

$D^\mathrm{VSV}_c = (K * w_c)/\rho$, no resampling. Same density map as Monte Carlo. Pooled PET-weighted runs. The pilot's 4 mm VSV is in `vsv_vs_mc.csv`.

In [ ]:
rows = []
vsv_seconds = {}
for s in ALL:
    for c in COMPARTMENTS:
        w = M.source_weights_v2(s, c, "pet")
        start = time.time()
        water_dose = engine(s)(w)
        vsv_seconds[s] = time.time() - start
        mc = d_c(s, c)
        mean_mc, mean_vsv = weighted(s, mc), weighted(s, water_dose / np.maximum(density_of[s], 0.1))
        for t in TARGETS:
            rows.append(dict(study_id=s, phantom=NAME[s], compartment=c, target=t, mc=mean_mc[t], vsv_native=mean_vsv[t]))
native = pd.DataFrame(rows)
native["ratio"] = native.vsv_native / native.mc
native.to_csv(DATA / "vsv_native_vs_mc.csv", index=False)
matrix = {s: native[native.study_id == s].pivot(index="target", columns="compartment", values="mc").reindex(index=TARGETS, columns=COMPARTMENTS) for s in ALL}
for s in ALL:
    matrix[s].to_csv(DATA / s / "dose_matrix_10c.csv")
own = {"spine": "spine_L2_L4"}
self_rows = native[native.apply(lambda r: r.target == own.get(r.compartment, r.compartment), axis=1)]
pilot = pd.read_csv(OLD / "vsv_vs_mc.csv")
pilot = pilot[(pilot.variant == "pet") & (pilot.compartment == pilot.target)].assign(phantom=lambda d: d.study_id.map(NAME))
print("Self-dose, native VSV/rho over MC")
print(self_rows.pivot(index="compartment", columns="phantom", values="ratio").reindex(COMPARTMENTS).round(3).to_string())
print("\nSame, 4 mm VSV of the pilot")
print(pilot.pivot(index="compartment", columns="phantom", values="ratio").round(3).to_string())
for s in ALL:
    print("\nNative VSV/rho over MC, all pairs,", NAME[s])
    print(native[native.study_id == s].pivot(index="target", columns="compartment", values="ratio").reindex(index=TARGETS, columns=COMPARTMENTS).round(2).to_string())
print("\nConvolution seconds per source:", {NAME[s]: round(v, 1) for s, v in vsv_seconds.items()})

**Reading.** **The self-dose deficit of the pilot is gone.** Native VSV/ρ over Monte Carlo is 1.00-1.02 on every source and phantom (GI on the two coarse phantoms 0.98-0.99, spine 0.99-1.00), against 0.76-0.98 with the 4 mm round trip; the third grid behaves like the others. A convolution takes 0.7-2.5 s.

Cross-doses are another matter. Within soft tissue they mostly fall within 10%. Three groups do not: **L2-L4 as a target, 0.53-0.74**, except from itself and lesions at times; **rest of body as a target, 1.05-1.25**; and pairs beyond 200 mm, 0 (glands to abdomen), which are of order 10⁻⁹ Gy/(MBq h). The first two look like a division by ρ that does not apply to photons, tested in section 6. Single pairs with lesions on few voxels (0.02-1.5) carry large statistical errors.

## 3. Rest-of-body halo

The new weight, the median, the activity moved, and what changes in the PET/uniform ratio and in the kidney cross-dose. Before is the pilot's `rest_of_body`, which also contained blood pool and spine; after is labels 14 and kidney fluid, without shells.

In [ ]:
rows = []
voxel_ml = {s: np.prod(spacing_of[s]) / 1000 for s in ALL}
for s in ALL:
    weights, median, shells, pet = M.rest_without_halo(s)
    rest = np.isin(organs_of[s], M.COMPARTMENTS_V2["rest_of_body"])
    any_shell = np.logical_or.reduce(list(shells.values()))
    row = dict(phantom=NAME[s], median_pct_ia_per_ml=median, rest_volume_l=rest.sum() * voxel_ml[s] / 1000, rest_pct_ia_before=pet[rest].sum() * voxel_ml[s], rest_pct_ia_after=weights.sum() * voxel_ml[s])
    for name, shell in shells.items():
        row[f"{name}_shell_ml"] = shell.sum() * voxel_ml[s]
        row[f"{name}_excess_pct_ia"] = (pet[shell] - median).sum() * voxel_ml[s]
    effective = lambda w: (w.sum() ** 2 / (w[w > 0] ** 2).sum()) * voxel_ml[s] / 1000
    row["effective_l_before"] = effective(np.where(rest, pet, 0.0))
    row["effective_l_after"] = effective(weights)
    rows.append(row)
halo = pd.DataFrame(rows)
halo.to_csv(DATA / "halo_summary.csv", index=False)
print(halo.set_index("phantom").T.round(4).to_string())

old_pet = pd.read_csv(OLD / "pooled_targets_mass_weighted.csv")
old_pet = old_pet[old_pet.compartment == "rest_of_body"].pivot_table(index=["study_id", "target"], columns="variant", values="D")
rows = []
for s in STUDIES:
    new_pet, new_uniform = weighted(s, d_c(s, "rest_of_body", "pet")), weighted(s, d_c(s, "rest_of_body", "uniform"))
    for t in TARGETS:
        rows.append(dict(phantom=NAME[s], target=t, pet_before=old_pet.loc[(s, t), "pet"], uniform_before=old_pet.loc[(s, t), "uniform"], pet_after=new_pet[t], uniform_after=new_uniform[t]))
change = pd.DataFrame(rows)
change["ratio_before"] = change.pet_before / change.uniform_before
change["ratio_after"] = change.pet_after / change.uniform_after
change["pet_after_over_before"] = change.pet_after / change.pet_before
change.to_csv(DATA / "halo_change.csv", index=False)
for k in ("ratio_before", "ratio_after", "pet_after_over_before"):
    print("\n", k, ": rest-of-body source, per target")
    print(change.pivot(index="target", columns="phantom", values=k).reindex(TARGETS).round(2).to_string())

rows = []
for s in STUDIES:
    tia, lesion_tia, _ = M.test_tia_v2(s)
    g = matrix[s]
    total = {t: sum(tia[c] * g.loc[t, c] for c in tia) + sum(lesion_tia.values()) * g.loc[t, "lesions"] for t in TARGETS}
    rows.append(dict(phantom=NAME[s], kidney_from_rest_before_gy_per_mbq_h=old_pet.loc[(s, "kidneys_functional"), "pet"], kidney_from_rest_after_gy_per_mbq_h=g.loc["kidneys_functional", "rest_of_body"], share_of_kidney_dose_after=tia["rest_of_body"] * g.loc["kidneys_functional", "rest_of_body"] / total["kidneys_functional"], kidney_test_dose_gy=total["kidneys_functional"]))
print(pd.DataFrame(rows).round(9).to_string(index=False))

rows = []
for s in ALL:
    weights, _, _, pet = M.rest_without_halo(s)
    kidney = masks_of[s]["kidneys_functional"]
    rho = np.maximum(density_of[s], 0.1)
    kidney_dose = lambda w: float(((engine(s)(w / w.sum()) / rho)[kidney] * density_of[s][kidney]).sum() / density_of[s][kidney].sum())
    old_w = np.where(np.isin(organs_of[s], M.COMPARTMENTS["rest_of_body"]), pet, 0.0)
    labels_w = np.where(np.isin(organs_of[s], M.COMPARTMENTS_V2["rest_of_body"]), pet, 0.0)
    row = dict(phantom=NAME[s])
    for name, w in (("pilot_source", old_w), ("blood_spine_removed", labels_w), ("halo_removed", weights)):
        row[f"{name}_pct_ia"] = w.sum() * voxel_ml[s]
        row[f"{name}_gy_per_mbq_h"] = kidney_dose(w)
        row[f"{name}_per_100_pct_ia_pet"] = kidney_dose(w) * w.sum() * voxel_ml[s]
    rows.append(row)
attribution = pd.DataFrame(rows)
attribution.to_csv(DATA / "halo_attribution_vsv.csv", index=False)
print("\nNative VSV/rho, kidney from the rest source, attribution of the change")
print(attribution.set_index("phantom").T.to_string(float_format=lambda v: f"{v:.3g}"))

**Reading.** **The halo was almost all of the rest-to-kidney cross-dose.** D(kidney ← rest), Monte Carlo, Gy/(MBq h): 9.80, 3.35, 6.67 ×10⁻⁷ before, 2.15, 1.97, 1.94 ×10⁻⁷ after, so ×0.22, 0.59, 0.29, and three phantoms that differed threefold now agree within 11%. PET/uniform on the kidneys: 6.3, 1.9, 3.9 becomes 1.36, 1.22, 1.18. The attribution with the VSV settles the cause: separating blood and spine moves it by under 3%, removing the shells gives the whole change (third grid 1.12 ×10⁻⁶ to 1.91 ×10⁻⁷).

**What it changes is the activity budget, not the kidney dose.** The kidney shell held 4.25, 0.63, 2.20 and 5.87 %IA that would have been counted twice; rest of body goes from 40.5 to 35.2, 32.6 to 31.7, 47.2 to 43.6, 37.3 to 30.3 %IA (the glands and bladder shells hold 0.1-0.8 %IA each). The rest source gives only 0.1-0.6% of the kidney dose. Effective volume of the source doubles or more (12.9 to 27.6 L, third grid 6.1 to 24.3 L), except on high TLV (14.8 L unchanged, the source there is not shaped by these shells).

**The PET weight is not uniform yet, and should not be:** PET/uniform stays 1.6-1.8 on the liver, 1.25-3.2 on the spleen and 1.3-3.9 on lesions, from spill-out of those structures, which were not in the requested list. The 3.2 rests on a single uniform run of 2×10⁵. Their PBPK TIA comes from the mask, so this activity is not counted twice.

## 4. Per-lesion dose

$D(\text{lesion } j) = \sum_i \mathrm{TIA}_i (K*w_i)/\rho + \big(\sum_i \mathrm{TIA}_i\big) L_\text{lesions}$, with $L$ the long-range term of the grouped source (section 6). Also tested here: lesions with the native kernel only, no $L$, since the kernels reach ±200 mm. The model of each lesion $i$ needs no stored image, since a single convolution of $\sum_i \mathrm{TIA}_i w_i$ gives the sum.

Two checks against Monte Carlo:
- **Single lesions:** a run of 2×10⁵ decays in one lesion, on the 4 lesions chosen on the high-TLV phantom (largest, seventh, about 2 mL, about 0.5 mL) all 3 of the typical one and 3 on the third grid: self-dose and spread to the other lesions.
- **Per-lesion TIA on the high-TLV phantom:** a lesion-only source of 2×10⁶ decays with $T_\mathrm{eff}$ drawn per lesion, log-uniform in 30-130 h (a test vector, not a PBPK draw), against the model and against the grouped $D_\text{lesions}$ scaled by the total TIA.

In [ ]:
SIGMA_MM = 20.0
lesions_of = {s: M.load(s, "lesions") for s in ALL}
def lesion_mean(s, dose, i):
    m = lesions_of[s] == i
    return float((dose[m] * density_of[s][m]).sum() / density_of[s][m].sum())

rows = []
for s in STUDIES[:2] + [THIRD]:
    weights = M.lesion_weights(s)
    rho = np.maximum(density_of[s], 0.1)
    for path in sorted((DATA / s).glob("D_lesion*_single.nii.gz")):
        i = int(path.name.split("_")[1][6:])
        mc = read(path)
        w = np.zeros(organs_of[s].shape)
        w[weights[i][0]] = weights[i][1]
        vsv = engine(s)(w) / rho
        volume = weights[i][0][0].size * voxel_ml[s]
        rows.append(dict(phantom=NAME[s], lesion=i, volume_ml=volume, mc_self=lesion_mean(s, mc, i), vsv_native_self=lesion_mean(s, vsv, i), led=0.0853 / (density_of[s][lesions_of[s] == i].sum() * voxel_ml[s])))
single = pd.DataFrame(rows)
single["vsv_over_mc"] = single.vsv_native_self / single.mc_self
single["mc_over_led"] = single.mc_self / single.led
single.to_csv(DATA / "single_lesion_check.csv", index=False)
print(single.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

s = R.HIGH_TLV
tia_table = pd.read_csv(DATA / s / "test_tia.csv")
lesion_tia = {int(n.split("_")[1]): v for n, v in zip(tia_table.compartment, tia_table.tia_mbq_h) if n.startswith("lesion_")}
weights = M.lesion_weights(s)
source = np.zeros(organs_of[s].shape)
for i, (idx, w) in weights.items():
    source[idx] += lesion_tia[i] * w
total_lesion_tia = sum(lesion_tia[i] for i in weights)
direct = read(DATA / s / "D_lesions_only_direct_gy.nii.gz")
rho = np.maximum(density_of[s], 0.1)
vsv_part = engine(s)(source) / rho
split_part = split_of(s)(source, density_of[s])
w_grouped = M.source_weights_v2(s, "lesions", "pet")
mc_grouped = d_c(s, "lesions")
base_grouped = split_of(s)(w_grouped, density_of[s])
near_grouped = M.near_source(w_grouped, spacing_of[s])
estimates = {
    "VSV/rho + L, as specified": vsv_part + total_lesion_tia * M.long_range_term(mc_grouped, water(s, "lesions"), density_of[s], body_of[s], SIGMA_MM, spacing_of[s]),
    "VSV/rho, no L": vsv_part,
    "split kernel, no L": split_part,
    "split + L by material": split_part + total_lesion_tia * M.long_range_by_material(mc_grouped, base_grouped, density_of[s], body_of[s], SIGMA_MM, spacing_of[s]),
    "split + L by material, near excluded": split_part + total_lesion_tia * M.long_range_by_material(mc_grouped, base_grouped, density_of[s], body_of[s], SIGMA_MM, spacing_of[s], near_grouped),
    "grouped MC x total TIA": total_lesion_tia * mc_grouped,
}
rows = []
for i in weights:
    voxels_i = weights[i][0][0].size
    rows.append(dict(lesion=i, volume_ml=voxels_i * voxel_ml[s], tia=lesion_tia[i], expected_decays=2e6 * lesion_tia[i] / total_lesion_tia, direct_gy=lesion_mean(s, direct, i), **{name: lesion_mean(s, e, i) / lesion_mean(s, direct, i) for name, e in estimates.items()}))
per = pd.DataFrame(rows)
per.to_csv(DATA / "per_lesion_check.csv", index=False)
per["decays"] = pd.cut(per.expected_decays, [0, 300, 1000, 3000, 10000, 1e8], labels=["<300", "300-1k", "1k-3k", "3k-10k", ">10k"])
show = ["VSV/rho + L, as specified", "split kernel, no L", "grouped MC x total TIA"]
summary = per.groupby("decays", observed=True).agg(n=("lesion", "size"), median_ml=("volume_ml", "median"), **{f"{name[:14]} p{q}": (name, (lambda q: lambda v: v.quantile(q / 100))(q)) for name in show for q in (5, 50, 95)})
print("Lesion mean over direct, by decays expected in the lesion in the direct run")
print(summary.round(3).to_string())
for name in estimates:
    print(f"{name:40s} median |error| {(per[name] - 1).abs().median():.3f}  max {(per[name] - 1).abs().max():.3f};  above 3000 decays: max {(per[per.expected_decays > 3000][name] - 1).abs().max():.3f}")
reference = weighted(s, direct)
organ_means = pd.DataFrame({name: {t: v / reference[t] for t, v in weighted(s, e).items()} for name, e in estimates.items()})
organ_means.insert(0, "direct_gy", pd.Series(reference))
organ_means.to_csv(DATA / "lesions_only_organ_means.csv")
print("\nLesion source alone, target mean over direct")
print(organ_means.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.semilogx(per.expected_decays, per["grouped MC x total TIA"], "o", ms=6, color=ORANGE, mec="white", mew=.8, label="grouped D, total TIA")
ax.semilogx(per.expected_decays, per["split kernel, no L"], "o", ms=6, color=BLUE, mec="white", mew=.8, label="native kernel per lesion, no long-range term")
ax.axhline(1, color=INK, lw=1)
ax.set(xlabel="decays in the lesion in the direct run", ylabel="dose ÷ direct Monte Carlo", title="High-TLV phantom, per-lesion TIA: each dot is a lesion")
ax.grid(color=GRID, lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(QC / "per_lesion.png", dpi=150); plt.show()

**Reading.** **Native VSV per lesion works and is necessary.** Single lesions from 0.13 to 813 mL: VSV/MC 0.987-1.017, on all three grids. With a per-lesion TIA on the high-TLV phantom (93 lesions), the grouped $D$ errs by a median of 27% and up to 130%; the native kernel by a median of 2.6%, and 4.4% at most on the 28 lesions with over 3000 decays in the direct run. The largest errors, up to 35%, are in lesions below 300 decays: that is the noise of the reference, and the single-lesion runs agree on those sizes. Mean over all lesions 1.00.

**On the other targets, the grouped long-range term as specified makes things worse:** with the lesion source alone, 0.78-1.02 against the direct run, where the plain VSV/ρ gives 0.79-1.22 and is closer on kidneys, spleen, glands and blood. The cause is the residual inside the lesions. A 1% difference on 4.7 Gy is 0.05 Gy, above the 0.038 Gy that the kidneys receive, and the low-pass spreads it to the neighbours. Splitting the kernel without $L$ gives 0.92-1.17: it overestimates far targets such as liver, bladder and rest of body (water around the source, a finite body in reality). Neither fix is clean for this source alone, but it matters little on the total, since it contributes at most 9% to these targets even on this phantom. Section 6 decides.

## 5. Blood pool and spine as sources

Self and main cross-doses of the new sources; dose to red marrow and blood with the test TIA: how much is now theirs and how much comes from elsewhere. Statistics from the batches of the new sources.

In [ ]:
for s in ALL:
    print(NAME[s], "Gy/(MBq h), rows target")
    print(matrix[s].to_string(float_format=lambda v: f"{v:.2e}"))
rows = []
for s in ALL:
    tia, lesion_tia, _ = M.test_tia_v2(s)
    g = matrix[s]
    tia_all = dict(tia, lesions=sum(lesion_tia.values()))
    for t in ("spine_L2_L4", "blood_pool"):
        parts = {c: tia_all[c] * g.loc[t, c] for c in COMPARTMENTS}
        total = sum(parts.values())
        top = sorted(parts.items(), key=lambda kv: -kv[1])[:3]
        rows.append(dict(phantom=NAME[s], target=t, test_dose_gy=total, **{f"top{k + 1}": f"{name} {v / total:.0%}" for k, (name, v) in enumerate(top)}))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

batches = raw[raw.kind == "pet"].copy()
batches["compartment"] = batches.run_id.str.split("/").str[1].str.replace("D_", "").str.replace("_pet", "")
stat = batches.groupby(["study_id", "compartment", "target"]).gy_per_mbq_h.agg(lambda v: v.std(ddof=1) / v.mean()).rename("rel_sd_2e5").reset_index()
stat["se_pct_5e5"] = 100 * stat.rel_sd_2e5 * np.sqrt(2e5 / 5e5)
for s in ALL:
    print("\n", NAME[s], "relative SE (%) at 5e5 decays, new sources")
    print(stat[stat.study_id == s].pivot(index="target", columns="compartment", values="se_pct_5e5").reindex(index=TARGETS).round(2).to_string())

**Reading.** **The separation was needed for the marrow.** L2-L4 receives 62-85% from its own source and the rest from elsewhere: kidneys 9-22%, rest of body 5-11%, lesions 8% on high TLV. Blood pool 77-85% from its own, rest of body 8-9%. The pilot had both inside rest of body: separating them moves the kidney cross-dose by under 3% (section 3), so the reason is the PBPK and the marrow dose, not the kidneys. Statistics at 5×10⁵ on the 3 phantoms: self-dose 0.04-0.65%; cross-doses to organs mostly 1-7%, above 10% only on doses of order 10⁻⁸ Gy/(MBq h) (glands, small lesions).

## 6. Voxel target

$$\hat D(\mathbf{x}) = \frac{K * \big(\sum_c \mathrm{TIA}_c w_c + \sum_i \mathrm{TIA}_i w_i\big)}{\rho(\mathbf{x})} + \sum_c \mathrm{TIA}_c L_c(\mathbf{x})$$

$L_c$ is the long-range term: $\rho D^\mathrm{MC}_c - K*w_c$, low-pass filtered with a Gaussian of width $\sigma$, divided by the low-pass of the mass, inside the body. Only the $L_c$ need Monte Carlo; the first term is one FFT per draw.

Reference: the direct run of each phantom, 2 independent halves of 10⁶ decays, same TIA vector (`test_tia.csv`, not a PBPK draw). The two halves give the noise ceiling of a voxel correlation: with the pooled reliability $R$ from the half correlation by Spearman-Brown, a noise-free estimate cannot exceed $\sqrt{R}$.

Compared on all 4 phantoms:
- **As specified:** VSV/ρ + $L$, $\sigma$ = 10, 20, 40 mm, and native VSV alone.
- **Split kernel, by material:** the kernel within 10 mm divided by ρ (electrons), the tail beyond 10 mm left as dose (photons: the mass absorption does not follow ρ). $L_c$ is the mass-weighted low-pass of $D^\mathrm{MC}_c$ minus that base, within lung, soft tissue and bone separately (ρ edges 0.5 and 1.15 g/mL), so a sharp material boundary survives the filter.
- **Near excluded:** the same, leaving out of the filter the voxels within 10 mm of the source, whose residual is self-dose and not long range. Not for the rest of body, which covers most of the body.
- **Rest 40:** $\sigma$ = 40 mm for the rest-of-body source only, whose decays are spread over the whole body, and the given $\sigma$ for the other sources.
- **Lesions VSV only:** each of these with $L$ dropped for the lesion source and kept for the organ sources.
- $\sum \mathrm{TIA}_c D_c$ with raw Monte Carlo $D_c$ (lesions grouped); on the typical phantom, $L_c$ from a single run of 2×10⁵ per compartment.

In [ ]:
SIGMAS = [10.0, 20.0, 40.0]
results, voxels, profile_store, long_seconds = [], [], {}, {}
PROFILE = "split+material, sigma 10, rest 40, lesions VSV only"
for s in ALL:
    tia, lesion_tia, _ = M.test_tia_v2(s)
    weights = M.lesion_weights(s)
    density, body, spacing = density_of[s], body_of[s], spacing_of[s]
    rho = np.maximum(density, 0.1)
    amounts = dict(tia, lesions=sum(lesion_tia[i] for i in weights))
    present = [c for c in COMPARTMENTS if amounts.get(c, 0) > 0]
    source = sum(tia[c] * M.source_weights_v2(s, c, "pet") for c in present if c != "lesions")
    for i, (idx, w) in weights.items():
        source[idx] += lesion_tia[i] * w
    start = time.time()
    vsv = engine(s)(source) / rho
    draw_seconds = time.time() - start
    split = split_of(s)(source, density)
    a, b = read(DATA / s / "D_direct_a_gy.nii.gz"), read(DATA / s / "D_direct_b_gy.nii.gz")
    direct = (a + b) / 2
    raw_mc = sum(amounts[c] * d_c(s, c) for c in present)
    sums = {}
    def add(name, c, term):
        part = "lesions" if c == "lesions" else "organs"
        sums.setdefault(name, {}).setdefault(part, np.zeros(direct.shape, np.float32))
        sums[name][part] += (amounts[c] * term).astype(np.float32)
    start = time.time()
    for c in present:
        w = M.source_weights_v2(s, c, "pet")
        mc = d_c(s, c)
        for sigma in SIGMAS:
            add(f"specified, sigma {sigma:.0f}", c, M.long_range_term(mc, water(s, c), density, body, sigma, spacing))
        base = split_of(s)(w, density)
        near = M.near_source(w, spacing)
        material = {(sigma, excluded): M.long_range_by_material(mc, base, density, body, sigma, spacing, near if excluded else None).astype(np.float32) for sigma in SIGMAS for excluded in (False, True)}
        wide = c == "rest_of_body"
        for sigma in SIGMAS:
            add(f"split+material, sigma {sigma:.0f}", c, material[sigma, False])
            add(f"split+material near-excl, sigma {sigma:.0f}", c, material[sigma, True])
        for sigma in SIGMAS[:2]:
            add(f"split+material, sigma {sigma:.0f}, rest 40", c, material[40.0, False] if wide else material[sigma, False])
            add(f"split+material near-excl, sigma {sigma:.0f}, rest 40", c, material[40.0, False] if wide else material[sigma, True])
        del material
        if s == R.TYPICAL:
            low = read(DATA / s / f"D_low_{c}.nii.gz")
            add("split+material, sigma 10, rest 40, L from 2e5", c, M.long_range_by_material(low, base, density, body, 40.0 if wide else 10.0, spacing))
            add("split+material, sigma 20, L from 2e5", c, M.long_range_by_material(low, base, density, body, 20.0, spacing))
            add("specified, sigma 20, L from 2e5", c, M.long_range_term(low, water(s, c), density, body, 20.0, spacing))
        del mc, base, near, w
    long_seconds[s] = time.time() - start
    estimates = {"VSV/rho only": lambda: vsv, "split kernel only": lambda: split, "sum of raw MC D_c": lambda: raw_mc}
    for name, parts in sums.items():
        first = vsv if name.startswith("specified") else split
        estimates[name] = (lambda first, parts: lambda: first + parts["organs"] + parts["lesions"])(first, parts)
        estimates[name + ", lesions VSV only"] = (lambda first, parts: lambda: first + parts["organs"])(first, parts)
    reference = weighted(s, direct)
    half = weighted(s, a), weighted(s, b)
    regions = {t: masks_of[s][t] for t in ("kidneys_functional", "liver", "spleen", "salivary_glands", "bladder", "gi_tract", "lesions", "spine_L2_L4", "blood_pool")}
    regions["body"] = body
    ceilings = {}
    for region, mask in regions.items():
        m = mask & (density > 0.5)
        if m.sum() >= 20:
            half_r = float(np.corrcoef(a[m], b[m])[0, 1])
            ceilings[region] = (m, half_r, np.sqrt(2 * half_r / (1 + half_r)) if half_r > 0 else np.nan)
    for name, build in estimates.items():
        estimate = build()
        means = weighted(s, estimate)
        for t in TARGETS:
            if reference[t] == reference[t]:
                results.append(dict(phantom=NAME[s], estimate=name, target=t, gy=means[t], direct_gy=reference[t], ratio=means[t] / reference[t], half_difference_pct=100 * abs(half[0][t] - half[1][t]) / (half[0][t] + half[1][t])))
        for region, (m, half_r, ceiling) in ceilings.items():
            voxels.append(dict(phantom=NAME[s], region=region, voxels=int(m.sum()), half_r=half_r, ceiling=ceiling, estimate=name, r=float(np.corrcoef(estimate[m], direct[m])[0, 1])))
        if name == PROFILE and s == R.TYPICAL:
            profile_store[s] = (direct, vsv.copy(), estimate)
    del sums, estimates, a, b, raw_mc, split
    print(NAME[s], f"one draw: FFT {draw_seconds:.1f} s; all long-range terms {long_seconds[s]:.0f} s for {len(present)} sources", flush=True)
results = pd.DataFrame(results); voxels = pd.DataFrame(voxels)
results.to_csv(DATA / "voxel_target_organ_means.csv", index=False)
voxels.to_csv(DATA / "voxel_target_correlation.csv", index=False)

score = results.assign(error=lambda d: (d.ratio - 1).abs())
main = score[~score.estimate.str.contains("2e5")]
table = main.groupby("estimate").agg(worst=("error", "max"), median=("error", "median"), above_2pct=("error", lambda v: int((v > .02).sum())))
for t in ("spine_L2_L4", "blood_pool", "gi_tract"):
    table[f"worst_{t}"] = main[main.target == t].groupby("estimate").error.max()
print("Absolute error of target means against the direct run, over 4 phantoms and 10 targets (40 values)")
print(table.sort_values("worst").round(3).to_string())
print("\nHalf difference of the direct run, %: largest per target")
print(score.groupby("target").half_difference_pct.max().reindex(TARGETS).round(2).to_string())
chosen = ["VSV/rho only", "specified, sigma 20", "split kernel only", "split+material, sigma 20", "split+material, sigma 20, lesions VSV only", "split+material, sigma 10, lesions VSV only", "split+material near-excl, sigma 10, rest 40, lesions VSV only", "split+material, sigma 10, rest 40, lesions VSV only", "sum of raw MC D_c"]
for s in ALL:
    g = results[results.phantom == NAME[s]]
    print("\nTarget mean over direct,", NAME[s])
    print(g.pivot(index="target", columns="estimate", values="ratio").reindex(index=TARGETS)[chosen].round(3).join(g.drop_duplicates("target").set_index("target")[["direct_gy", "half_difference_pct"]]).to_string())
low_cols = [c for c in results.estimate.unique() if "2e5" in c]
print("\nTypical phantom, L from one run of 2e5 per source")
print(results[results.estimate.isin(low_cols)].pivot(index="target", columns="estimate", values="ratio").reindex(TARGETS).round(3).T.to_string())
voxel_cols = ["VSV/rho only", "specified, sigma 20", "split+material, sigma 10, lesions VSV only", "split+material, sigma 10, rest 40, lesions VSV only", "sum of raw MC D_c"]
for s in ALL:
    g = voxels[voxels.phantom == NAME[s]]
    print("\nVoxel r with direct,", NAME[s])
    print(g.pivot(index="region", columns="estimate", values="r")[voxel_cols].round(3).join(g.drop_duplicates("region").set_index("region")[["half_r", "ceiling", "voxels"]].round(3)).to_string())

In [ ]:
s = STUDIES[0]
direct, vsv, target = profile_store[s]
kidney = masks_of[s]["kidneys_functional"]
x, y, _ = [int(np.median(i)) for i in np.where(kidney)]
z = (np.arange(direct.shape[2]) - np.median(np.where(kidney)[2])) * spacing_of[s][2]
window = lambda a: a[x - 6:x + 7, y - 6:y + 7, :].mean(axis=(0, 1))
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.semilogy(z, window(direct), color=INK, lw=1.2, label="direct MC, 2e6")
ax.semilogy(z, np.maximum(window(vsv), 1e-6), color=ORANGE, lw=2, label="native VSV/ρ")
ax.semilogy(z, np.maximum(window(target), 1e-6), color=BLUE, lw=2, label="voxel target, σ 10 mm, rest 40")
ax.set(xlabel="axial distance from the kidney centre (mm)", ylabel="Gy, test TIA", title=f"Axial profile through the kidney, {NAME[s]} phantom")
ax.grid(color=GRID, lw=.6); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(QC / "voxel_target_profile.png", dpi=150); plt.show()

fig, axes = plt.subplots(3, 1, figsize=(8, 6.4), sharex=True, sharey=True)
panels = ["specified, sigma 20", "split+material, sigma 20", "split+material, sigma 10, rest 40, lesions VSV only"]
for ax, name in zip(axes, panels):
    g = results[results.estimate == name]
    ax.axhspan(.98, 1.02, color=GRID, zorder=0)
    ax.axhline(1, color=INK, lw=1)
    for (phantom, part), color in zip(g.groupby("phantom"), [GREEN, ORANGE, "#6e5bb8", BLUE]):
        x = [TARGETS.index(v) for v in part.target]
        ax.scatter(x, part.ratio, s=42, color=color, edgecolor="white", label=phantom, zorder=3)
    ax.set_title(name, loc="left")
    ax.set_ylabel("target ÷ direct")
axes[0].legend(frameon=False, ncol=4, loc="lower left")
axes[-1].set_xticks(range(len(TARGETS)), [t.replace("_functional", "").replace("_glands", "").replace("_tract", "").replace("spine_L2_L4", "L2-L4") for t in TARGETS], rotation=30, ha="right")
fig.tight_layout(); fig.savefig(QC / "target_means_by_variant.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
regions = ["kidneys_functional", "liver", "spleen", "salivary_glands", "bladder", "gi_tract", "lesions", "spine_L2_L4"]
for ax, s in zip(axes, ALL):
    g = voxels[voxels.phantom == NAME[s]].set_index(["region", "estimate"])
    present_regions = [r for r in regions if (r, PROFILE) in g.index]
    positions = np.arange(len(present_regions))
    ceiling = [g.loc[(r, PROFILE), "ceiling"] for r in present_regions]
    ax.scatter(positions, ceiling, marker="_", s=120, color=INK, label="noise ceiling", zorder=2)
    ax.scatter(positions, [g.loc[(r, PROFILE), "r"] for r in present_regions], s=42, color=BLUE, edgecolor="white", label="voxel target", zorder=3)
    ax.scatter(positions, [g.loc[(r, "sum of raw MC D_c"), "r"] for r in present_regions], s=42, color=ORANGE, edgecolor="white", label="raw MC D_c", zorder=3)
    ax.set_xticks(positions, [r.replace("_functional", "").replace("_glands", "").replace("_tract", "").replace("spine_L2_L4", "L2-L4") for r in present_regions], rotation=45, ha="right")
    ax.set_title(NAME[s], loc="left"); ax.grid(axis="y", color=GRID, lw=.6); ax.set_axisbelow(True)
axes[0].set_ylabel("voxel r with the direct run"); axes[0].legend(frameon=False, loc="lower left")
fig.tight_layout(); fig.savefig(QC / "voxel_correlation.png", dpi=150); plt.show()

### Voxel noise of the long-range term

Organ means and the correlation with the direct run cannot see noise in a term this small. So: on the typical phantom, the voxel difference between $L$ built from one run of 2×10⁵ per source and from the pooled run of 10⁶, organ sources only.

In [ ]:
s = R.TYPICAL
tia, lesion_tia, _ = M.test_tia_v2(s)
weights = M.lesion_weights(s)
density, body, spacing = density_of[s], body_of[s], spacing_of[s]
source = sum(tia[c] * M.source_weights_v2(s, c, "pet") for c in tia if tia[c] > 0 and c != "lesions")
for i, (idx, w) in weights.items():
    source[idx] += lesion_tia[i] * w
base = split_of(s)(source, density)
organ_sources = [c for c in COMPARTMENTS if c != "lesions" and tia.get(c, 0) > 0]
NOISE_VARIANTS = {"sigma 10": (10.0, 10.0, False), "sigma 20": (20.0, 20.0, False), "sigma 10, rest 40": (10.0, 40.0, False), "sigma 20, rest 40": (20.0, 40.0, False), "near-excl sigma 10, rest 40": (10.0, 40.0, True), "near-excl sigma 20, rest 40": (20.0, 40.0, True)}
terms = {}
for c in organ_sources:
    w = M.source_weights_v2(s, c, "pet")
    b, near = split_of(s)(w, density), M.near_source(w, spacing)
    mc, low_mc = d_c(s, c), read(DATA / s / f"D_low_{c}.nii.gz")
    for key in {(sr if c == "rest_of_body" else so, n and c != "rest_of_body") for so, sr, n in NOISE_VARIANTS.values()}:
        sigma, use_near = key
        terms[(c, key)] = tuple((tia[c] * M.long_range_by_material(x, b, density, body, sigma, spacing, near if use_near else None)).astype(np.float32) for x in (mc, low_mc))
rows = []
region_masks = {**{t: masks_of[s][t] for t in ("kidneys_functional", "liver", "spleen", "salivary_glands", "gi_tract", "blood_pool", "spine_L2_L4", "rest_of_body", "lesions")}, "body": body}
for name, (so, sr, n) in NOISE_VARIANTS.items():
    picks = [terms[(c, (sr, False) if c == "rest_of_body" else (so, n))] for c in organ_sources]
    full = sum(p[0] for p in picks)
    difference = sum(p[1] for p in picks) - full
    target = base + full
    for region, mask in region_masks.items():
        m = mask & (density > 0.5)
        rows.append(dict(variant=name, region=region, rms_over_median_pct=100 * np.sqrt(np.mean(difference[m] ** 2)) / np.median(target[m]), mean_bias_pct=100 * difference[m].mean() / target[m].mean()))
noise = pd.DataFrame(rows)
noise.to_csv(DATA / "long_range_noise_2e5.csv", index=False)
print("Typical phantom. Voxel RMS of L from 2e5 minus L from 1e6, over the median target in the region, %. Scales as 1/sqrt(n): x0.63 at 5e5.")
print(noise.pivot(index="region", columns="variant", values="rms_over_median_pct")[list(NOISE_VARIANTS)].round(2).to_string())

**Reading.** **The target as specified fails on the spine.** VSV/ρ + $L$, σ 20 mm: L2-L4 0.88-0.97 and blood pool 0.95-1.01, against raw Monte Carlo 0.98-1.02 and 0.98-1.01 on the same run. Smaller σ helps little (worst 9.6% at 10 mm, 17.3% at 40 mm). The reason is that the error has a sharp edge at the bone: photon dose does not follow 1/ρ and is higher in bone, and a low-pass across soft tissue and bone removes that. The bone fraction of L2-L4 is 51-94% of the mass.

**Variants, 40 target means on 4 phantoms against the direct run:**

| variant | worst | median | above 2% | worst L2-L4 | worst blood |
|---|---|---|---|---|---|
| VSV/ρ only | 15.4% | 1.0% | 10 | 15.4% | 2.4% |
| specified, σ 20 | 12.4% | 0.6% | 8 | 12.4% | 5.0% |
| split + material, σ 20 | 4.2% | 0.6% | 5 | 3.8% | 4.2% |
| split + material, σ 10, lesions VSV only | 3.0% | 0.5% | 3 | 3.0% | 2.2% |
| **split + material, σ 10, rest 40, lesions VSV only** | **3.4%** | **0.5%** | **3** | **3.4%** | **2.0%** |
| split + material near-excl, σ 10, lesions VSV only | 2.9% | 0.9% | 3 | 2.9% | 1.9% |
| raw Monte Carlo $D_c$, lesions grouped | 2.4% | 0.3% | 3 | 2.4% | 2.1% |

**Material split:** the deficit on the spine comes down from 12.4% to 3.8%, and most of that is the filter by material, since the split kernel alone still has 10.6%. **Lesions VSV only:** better than keeping the lesion term on the blood pool, 4.2% to 2.1% at σ 20 and 3.8% to 2.2% at σ 10, nothing lost elsewhere: the term goes, and the kernels of ±200 mm stay in charge.

**Choice: split kernel, $L$ by material, σ 10 mm, σ 40 mm for the rest-of-body source, lesions with native VSV only.** The variants in bold and just above it, and near-excl, are equivalent on the direct run: they differ by 0.5 points at worst, on L2-L4 on one phantom, where the two halves of the reference themselves differ by up to 1.4%. The noise, which the direct run cannot see, separates them: at 2×10⁵ the rest-of-body voxels carry 18.6% with σ 10, 8.1% with rest 40; blood pool 5.8% against 1.7%. Near excluded would lower it further (rest of body 5.6%, organs under 1%, but L2-L4 as before) at the price of leaving +1% on kidneys and liver, a median of 0.9% and 5 values above 2% in the rest-40 version. That is a defensible alternative, not a better one by the criterion set.

On the chosen variant: kidneys 1.00, liver 1.00-1.01, spleen 0.98-1.01, glands 0.99-1.02, bladder 1.00-1.01, GI 0.99-1.00, blood 0.98-1.02, **L2-L4 0.97-1.00**, rest of body 1.00-1.01, lesions 0.99-1.01. The third grid works like the others, its worst being L2-L4 at 0.966.

**Voxel correlation does not discriminate variants, only against raw Monte Carlo.** Over the body 0.943, 0.952, 0.983, 0.967 against ceilings of 0.944, 0.955, 0.983, 0.967; in organs a median of 0.004 below the ceiling, while raw $D_c$ with lesions grouped lose 0.047, up to 0.29 (lesions of high TLV 0.625 against 0.915). Every variant, VSV alone included, gives the same r to the third decimal: the correlation is set by the first term and by the noise of the reference. It says the target is as good as a 2×10⁶ run can show, and no more. Where the ceiling is low (L2-L4 0.27-0.60, blood 0.20-0.42) it says little.

**With $L$ from one run of 2×10⁵ per source, on the typical phantom, target means stay within 1%** (L2-L4 0.999, blood 1.006, GI 0.992, spleen 1.009): means do not ask for a larger budget, voxel noise does. Third-grid spleen 0.98 and glands 1.02 appear in raw Monte Carlo as well (0.981, 1.017), while the halves differ by only 0.2%: not a defect of the target, cause not found.

## 7. Cost of the full split

Per phantom: 10 GATE runs of $n$ decays each; kernels once per voxel size; the long-range terms and their storage; one FFT and one sum per PBPK draw. Times at 4 threads, with the other session also running, from the runs of the pilot and of this notebook. Grid effect interpolated in the voxel count as in notebook 14.

All three kernels now exist, so the full run has no kernel cost. The rate of the pilot and the rate measured here, with another session on the machine, bound the estimate.

In [ ]:
timing = pd.read_csv(OLD / "targets.csv").drop_duplicates(["study_id", "compartment", "variant", "batch"])
tests = timing[timing.variant == "timing"].set_index("study_id")
per_batch = timing[(timing.variant == "pet") & (timing.compartment != "combined")]
rate = {s: (2e5 - 1e4) / (per_batch[per_batch.study_id == s].seconds.mean() - tests.loc[s, "seconds"]) for s in STUDIES}
init = {s: tests.loc[s, "wall_seconds"] - 1e4 / rate[s] for s in STUDIES}
new_rate = raw[raw.kind.isin(["pet", "uniform"])].drop_duplicates("run_id").assign(rate=lambda d: d.n / (d.seconds - d.study_id.map(init)))
print("Decays/s, pilot:", {NAME[s]: round(rate[s]) for s in STUDIES}, "| this notebook, median:", new_rate.groupby("study_id").rate.median().rename(index=NAME).round(0).to_dict())
summary = pd.read_csv(M.ph.psma_data.PROCESSED / "phantoms_summary.csv")
split = pd.read_csv(M.ph.psma_data.PROCESSED / "psma_68ga_split.csv").merge(summary[["study_id", "n_lesions"]], on="study_id")
grid = lambda s: json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())["grid"]
split["voxels"] = split.study_id.map(lambda s: int(np.prod(grid(s)["shape"])))
split["size"] = split.study_id.map(lambda s: "x".join(f"{v:.2f}" for v in grid(s)["spacing_mm"]))
print(split.groupby(["split", "size"]).agg(studies=("study_id", "size"), voxels=("voxels", "median"), lesions_median=("n_lesions", "median"), lesions_max=("n_lesions", "max")).to_string())
ref = [np.prod(grid(s)["shape"]) for s in STUDIES]
per_decay = np.polyfit(ref, [1 / rate[s] for s in STUDIES], 1)
fit = np.polyfit(ref, [init[s] for s in STUDIES], 1)
rows = []
for n in (1e5, 2e5, 5e5, 1e6):
    def seconds(frame):
        return (10 * (np.polyval(fit, frame.voxels) + n * np.polyval(per_decay, frame.voxels))).sum()
    train = split[split.split == "train"]
    rows.append(dict(n_per_source=n, minutes_fine=10 * (np.polyval(fit, 17235968) + n * np.polyval(per_decay, 17235968)) / 60, minutes_coarse=10 * (np.polyval(fit, 5503680) + n * np.polyval(per_decay, 5503680)) / 60, days_train=seconds(train) / 86400, days_train_and_test=seconds(split) / 86400))
cost = pd.DataFrame(rows)
cost.to_csv(DATA / "cost_full_split.csv", index=False)
print(cost.round(2).to_string(index=False))
print("Kernels:", kernels[["voxel", "seconds"]].to_dict("records"))
under_load = raw[(raw.kind == "pet") & (raw.n == 2e5)].drop_duplicates("run_id").assign(voxels=lambda d: d.study_id.map(lambda s: np.prod(grid(s)["shape"])))
measured = under_load.groupby("study_id").agg(voxels=("voxels", "first"), median_s_per_2e5=("wall_seconds", "median")).rename(index=NAME)
print(measured.to_string())
slope = np.polyfit(measured.voxels, measured.median_s_per_2e5, 1)
days_load = {name: 10 * np.polyval(slope, frame.voxels).sum() / 86400 for name, frame in (("train", split[split.split == "train"]), ("train and test", split))}
print("Measured here, one 2e5 run per source and phantom, days:", {k: round(v, 2) for k, v in days_load.items()})
s = STUDIES[0]
coarse = {}
for c in COMPARTMENTS:
    w = M.source_weights_v2(s, c, "pet")
    term = M.long_range_by_material(d_c(s, c), split_of(s)(w, density_of[s]), density_of[s], body_of[s], 40.0 if c == "rest_of_body" else 10.0, spacing_of[s])
    factor = [max(1, int(round((16.0 if c == "rest_of_body" else 5.0) / v))) for v in spacing_of[s]]
    small = ndi.zoom(term, [1 / f for f in factor], order=1).astype(np.float32)
    path = DATA / s / f"tmp_long_{c}.nii.gz"
    nib.save(nib.Nifti1Image(small, np.eye(4)), path)
    coarse[c] = path.stat().st_size / 1e6
    path.unlink()
print("Long-range terms at about 5 mm (rest of body 16 mm), MB per phantom, typical:", round(sum(coarse.values()), 1), "| raw D_c on disk, MB per phantom, third grid:", round(sum(f.stat().st_size for f in (DATA / THIRD).glob("D_*_pet.nii.gz")) / 1e6))

**Reading and proposal.** **Proposed for the full run:** the 10 compartments, the shells replaced by the median, the three kernels already generated, and the chosen variant of section 6. Each phantom keeps $L_c$ for 10 sources; each draw of the cohort is one FFT plus a sum, 0.7-2.5 s on one phantom, whatever the number of lesions.

**Cost, 10 runs per phantom, 180 training and 227 with test:**

| decays per source | per phantom, fine / coarse grid | training | training + test |
|---|---|---|---|
| 2×10⁵ | 15 / 9 min | 1.5-2.6 days | 1.9-3.2 days |
| **5×10⁵** | **30 / 18 min** | **3.1-6 days** | **4.0-7.7 days** |
| 10⁶ | 56 / 33 min | 5.8 days upward | 7.4 days upward |

Lower bound: rates of the pilot. Upper bound: measured here with another session on the machine (2×10⁵, scaled by 2.4 at 5×10⁵ since initialisation does not scale). The third grid is as slow as the fine one. No further kernel cost. Building $L_c$ adds about 2 minutes per phantom on a single core, a few hours overall. Storage: $L_c$ at about 5 mm (rest of body 16 mm) 12 MB per phantom, raw $D_c$ 23 MB; about 8 GB with both, under 3 GB for the $L_c$ alone. Kernels 44 MB.

**I propose 5×10⁵.** Organ means do not need it, as 2×10⁵ passes them on the typical phantom, but voxel noise in the long-range term does: at 5×10⁵ the 8.1% in rest of body becomes about 5%, 3.2% on L2-L4 about 2%. Above that the gain is small, since the total error is already set by the systematic terms of section 6. The 3 tested phantoms need not be redone for the 7 compartments that did not change.

**The full run has not been launched.**

## Limitations

- **Variant chosen on the same 4 phantoms that test it**, with no independent set; 29 variants were compared. Differences below about 1 point are within the noise of the reference (halves differ by up to 1.7% on blood and bladder, 2.5% on glands, 3.4% on lesions with few decays).
- **Test TIA, not a PBPK draw:** the proportions between sources will change, and with them the weight of each error. The spine error will grow if the marrow source weighs less than it does here (62-85% of its dose).
- **Remaining bias: L2-L4 down to -3.4%, blood pool ±2%.** The direct run's own uncertainty there is 0.6-1.7%.
- Per-lesion TIA checked on one phantom only, against a single direct run with few decays in small lesions; **without $L$, the lesion source alone is +15-17% on liver, bladder and rest of body** (water around the source, finite body). In the total it counted for under 2% here; on a patient with a large burden and little else in the organ it could weigh more.
- The third grid has one phantom, 11 lesions, with truncated parotids (a frequent flag). The low budget was tested only on the typical phantom. Voxel noise of $L$ was measured against a run of 10⁶, itself noisy, so it is a slight overestimate.
- Edges chosen by us, not fitted: ρ 0.5 and 1.15 g/mL for the materials, 10 mm for the kernel split, σ for the rest of body. L2-L4 mixes bone and marrow in the voxel, so the dose is to the mixture, not to the marrow; the cross-dose ratios found in bone are not a marrow model.
- Median replacement in the shells assumes the true background there equals the median of rest of body; spill-out from liver, spleen and lesions stays in that source. Glands and bladder shells depend on the mask, and the glands are small in our masks (nb11).
- Costs measured with another session on the machine; full-run throughput is an estimate over the grids and not a test.